# Step 3 — Load into DuckDB
Load all transformed Parquet files into DuckDB analytical database

In [1]:
import duckdb
import pandas as pd
import os

TRANSFORMED = '../data/transformed'
DB_PATH     = '../analytics.duckdb'

# Connect to DuckDB (creates the file if not exists)
conn = duckdb.connect(DB_PATH)
print('Connected to DuckDB!')
print('Database path:', DB_PATH)

Connected to DuckDB!
Database path: ../analytics.duckdb


## Load Transformed Parquet Files into DuckDB

In [2]:
# Load orders_full table
conn.execute('''
    CREATE OR REPLACE TABLE orders_full AS
    SELECT * FROM read_parquet('../data/transformed/orders_full/*.parquet')
''')
count = conn.execute('SELECT COUNT(*) FROM orders_full').fetchone()[0]
print(f'orders_full loaded: {count:,} rows')

orders_full loaded: 96,478 rows


In [3]:
# Load monthly_revenue table
conn.execute('''
    CREATE OR REPLACE TABLE monthly_revenue AS
    SELECT * FROM read_parquet('../data/transformed/monthly_revenue/*.parquet')
    ORDER BY year, month
''')
count = conn.execute('SELECT COUNT(*) FROM monthly_revenue').fetchone()[0]
print(f'monthly_revenue loaded: {count:,} rows')

monthly_revenue loaded: 23 rows


In [4]:
# Load revenue_by_state table
conn.execute('''
    CREATE OR REPLACE TABLE revenue_by_state AS
    SELECT * FROM read_parquet('../data/transformed/revenue_by_state/*.parquet')
    ORDER BY total_revenue DESC
''')
count = conn.execute('SELECT COUNT(*) FROM revenue_by_state').fetchone()[0]
print(f'revenue_by_state loaded: {count:,} rows')

revenue_by_state loaded: 27 rows


In [5]:
# Load weather_impact table
conn.execute('''
    CREATE OR REPLACE TABLE weather_impact AS
    SELECT * FROM read_parquet('../data/transformed/weather_impact/*.parquet')
''')
count = conn.execute('SELECT COUNT(*) FROM weather_impact').fetchone()[0]
print(f'weather_impact loaded: {count:,} rows')

weather_impact loaded: 3 rows


## Verify Tables in DuckDB

In [6]:
# Show all tables
tables = conn.execute("SHOW TABLES").fetchdf()
print('Tables in DuckDB:')
print(tables)

Tables in DuckDB:
               name
0   monthly_revenue
1       orders_full
2  revenue_by_state
3    weather_impact


In [7]:
# Preview monthly revenue
df = conn.execute('SELECT * FROM monthly_revenue').fetchdf()
print('Monthly Revenue:')
df

Monthly Revenue:


,year,month,total_orders,total_revenue,avg_order_value,avg_delivery_days,late_deliveries
0,2016,9,1,NaN,NaN,55.0,1
1,2016,10,265,46566.71,175.72,19.6,2
2,2016,12,1,19.62,19.62,5.0,0
3,2017,1,750,127545.67,170.06,12.8,22
4,2017,2,1653,271298.65,164.13,13.3,49
5,2017,3,2546,414369.39,162.75,13.0,116
6,2017,4,2303,390952.18,169.76,15.0,151
7,2017,5,3546,567066.73,159.92,11.4,106
8,2017,6,3135,490225.60,156.37,12.0,95
9,2017,7,3872,566403.93,146.28,11.5,108


In [8]:
# Preview top 5 states by revenue
df = conn.execute('SELECT * FROM revenue_by_state LIMIT 5').fetchdf()
print('Top 5 States by Revenue:')
df

Top 5 States by Revenue:


,customer_state,total_orders,total_revenue,avg_order_value,avg_delivery_days
0,SP,40501,5770266.19,142.48,8.7
1,RJ,12350,2055690.45,166.45,15.2
2,MG,11354,1819277.61,160.23,11.9
3,RS,5345,861802.40,161.24,15.2
4,PR,4923,781919.55,158.83,11.9


In [9]:
# Preview weather impact
df = conn.execute('SELECT * FROM weather_impact').fetchdf()
print('Weather Impact on Orders:')
df

Weather Impact on Orders:


,rain_category,total_orders,avg_order_value
0,No Rain,33726,161.81
1,Heavy Rain,16667,159.63
2,Light Rain,46085,158.41


## Run Business Insight Queries

In [10]:
# Business Query 1: Total KPIs
kpis = conn.execute('''
    SELECT
        COUNT(order_id)              AS total_orders,
        ROUND(SUM(total_payment), 2) AS total_revenue,
        ROUND(AVG(total_payment), 2) AS avg_order_value,
        ROUND(AVG(delivery_days), 1) AS avg_delivery_days,
        SUM(was_late)                AS total_late_deliveries
    FROM orders_full
''').fetchdf()
print('KPIs:')
kpis

KPIs:


,total_orders,total_revenue,avg_order_value,avg_delivery_days,total_late_deliveries
0,96478,15422461.77,159.86,12.5,6534.0


In [11]:
# Business Query 2: Best month for sales
best_month = conn.execute('''
    SELECT year, month, total_orders, total_revenue
    FROM monthly_revenue
    ORDER BY total_revenue DESC
    LIMIT 5
''').fetchdf()
print('Top 5 Best Months:')
best_month

Top 5 Best Months:


,year,month,total_orders,total_revenue
0,2017,11,7289,1153528.05
1,2018,4,6798,1132933.95
2,2018,5,6749,1128836.69
3,2018,3,7003,1120678.00
4,2018,1,7069,1078606.86


In [11]:
conn.close()
print('=' * 50)
print('LOAD COMPLETE')
print('=' * 50)
print('DuckDB database ready at: ../analytics.duckdb')
print('Ready for Step 4 — Dashboard!')

LOAD COMPLETE
DuckDB database ready at: ../analytics.duckdb
Ready for Step 4 — Dashboard!
